# Lab 1: Modeling Pointing Time

A common approach in cognitive science is to collect data through experiments and use it to develop and test mathematical models of human behaviour. 
In this lab, we will practice this approach using simulated participants, building and evaluating a model that predicts how long it takes to point at a target.


## Part 0: Pointing behaviour

To click a button on a screen, you move the mouse pointer from its current position to the button and click inside it. We are interested in the **movement time**: how long it takes to complete this action.

This movement time mainly depends on two features of the task: the **distance (D)** to the target and the target’s **width (W)**.


<img src="pointing-behaviour.png" alt="A black mouse pointer and a target with a bold black border and centered label. A gray dashed path meets the target centerline, aligned with the D endpoint. The centered Target label masks the lines immediately behind it for readability. Green annotations label Distance (D) from the pointer tip to the target center and Width (W) across the target width." width="800" style="max-width: 100%; height: auto;">

### Task 1: Explain your expectations (~3 minutes)
**How would you expect distance and width to affect movement time?**

Discuss your expectations with your neighbour!

## Part 1: Installation and setup

We will be working in Jupyter Notebook, which lets you write and run code in cells. 
For help setting up the environment, see the [setup instructions](README.md).

**Jupyter tips for first-time users**

Click a code cell to select it, then run it using either of the following:
- Click the ▶ Run button in the toolbar.
- Press **Shift + Enter** to run the cell and move to the next one.

The output appears directly below the cell. Run the code cells in order, since later cells use variables created earlier. Save your progress regularly.
Jupyter remembers the code you have run, even if you later edit or delete it. After changing a cell, run it again, then rerun any cells that use its results.

If the notebook gets into an unexpected state, restart the kernel to clear its memory, then run the code cells again from the top.


In [ ]:
# Run this cell to load the simulator and the libraries used in this lab.
from lab1.mystery.mystery import PointingParticipant
from lab1.simulations import simulate_participant, simulate_group

import numpy as np
import pandas as pd
import seaborn as sns
from hyperopt import hp, fmin, tpe


print("Setup complete! Ready to begin Part 2.")


## Part 2: Understand your participants

In this lab, we will build a mathematical model to predict movement time in a pointing task. 
To collect data for building this model, we will conduct experiments using `PointingParticipant`, a class that simulates a person pointing at a target.

Like working with real person, we treat `PointingParticipant` as a black box, which means we can only learn about its behaviour through experiments. To run an experiment, call its `pointing_time` method with the experimental parameters `distance` and `width`. 

Try different values of `distance` and `width`. How does the pointing time change?


In [ ]:
participant = PointingParticipant(participant_id=0)
participant.pointing_time(distance = 400, width = 50) 

To better understand how the simulated participant performs, we can run multiple experiments with different distances and widths using the provided `simulate_participant` function. 
This function returns the results as a [pandas DataFrame](https://pandas.pydata.org/docs/getting_started/intro_tutorials/01_table_oriented.html), which is a convenient way to store and analyze tabular data.

In [ ]:
distances = range(10, 501, 10)  # Distances: 10, 20, ..., 500
widths = range(5, 51, 5)    # Widths: 5, 10, ..., 50
conditions = [(d, w) for d in distances for w in widths]

experiments_data = simulate_participant(participant, conditions, repeats=500)
experiments_data

### Task 2: Visualize the experimental data (~15 minutes)

Use [Seaborn](https://seaborn.pydata.org/) to create a visualization of the data above, showing the relationship between distance, width, and mean movement time. Briefly describe the patterns you observe.


Hint: [seaborn.heatmap](https://seaborn.pydata.org/generated/seaborn.heatmap.html) might be a good fit here.

In [ ]:
# TODO


## Part 3: Modeling and parameter selection

The results from the previous task suggest that pointing time increases with distance and decreases with target width. 
To capture these two trends, let’s start with a simple model for predicting movement time $T$:

$$
T = a\,\frac{D}{W} + b, \qquad a > 0 \text{ and } b > 0
$$

Here, $D$ is the distance and $W$ is the target width. The parameter $a$ controls how quickly pointing time increases with $D/W$, while $b$ sets the baseline time.

In [ ]:
class ExampleModel:
    def __init__(self, a, b):
        self.a = a
        self.b = b

    def pointing_time(self, distance, width):
        return self.a * (distance / width) + self.b


We need to choose values for $a$ and $b$ so that the predicted pointing times are close to the measured times. 

In [ ]:
model = ExampleModel(a=0.01, b=0.15)

d = 400
w = 50
predicted_time = model.pointing_time(d, w)
measured_time = participant.pointing_time(d, w)

print(
    f"Predicted: {predicted_time:.3f} s | "
    f"Measured: {measured_time:.3f} s | "
    f"Residual: {measured_time - predicted_time:.3f} s"
)


Run the cell below to collect experimental data and examine the comparison plot. 
`simulate_group` returns experimental data from a specified number of simulated participants. 
`plot_predictions(model, experiments_data)` compares the model’s predictions with the mean measured times for each distance and width combination.


In [ ]:
distances = range(10, 501, 10)  # Distances: 10, 20, ..., 500
widths = range(5, 51, 5)       # Widths: 5, 10, ..., 50
experiments_data = simulate_group(conditions, n_participants=100, repeats=8)

def plot_predictions(model, experiments_data):
    measured_data = (
        experiments_data.groupby(["distance", "width"], as_index=False)
        .agg(time=("pointing_time", "mean"))
    )

    comparison_data = measured_data.assign(
        predicted_time=[
            model.pointing_time(d, w)
            for d, w in zip(measured_data["distance"], measured_data["width"])
        ]
    )

    colors = sns.color_palette("YlGnBu", 2)
    chart = sns.FacetGrid(
        comparison_data, col="width", col_wrap=5, height=2.5,
    )

    chart.map_dataframe(
        sns.scatterplot, x="distance", y="time",
        color=colors[0], label="Measured", s=6, linewidth=0,
    )
    chart.map_dataframe(
        sns.scatterplot, x="distance", y="predicted_time",
        color=colors[1], label="Predicted", s=6, linewidth=0,
    )

    chart.add_legend()
    chart.set_axis_labels("", "").set_titles("Width = {col_name}")
    chart.figure.supxlabel("Distance")
    chart.figure.supylabel("Movement time")
    chart.figure.subplots_adjust(left=0.07, bottom=0.12)


plot_predictions(ExampleModel(a=0.01, b=0.15), experiments_data)

### Task 3: Manually adjust the model parameters (~5 minutes)

Adjust `a` and `b` and try to bring the model’s predictions as close as possible to the measured times. Reuse the function `plot_predictions` and the simulated data `experiments_data` to visualize the comparison.

In [ ]:
# TODO

## Part 4: Automated parameter search

The plot helps us get a sense of how well our model is performing. But in order to make a more objective comparison, we need a quantitative measure: **mean squared error (MSE)**.

$$
\mathrm{MSE} = \operatorname{mean}\!\left[(\text{measured time} - \text{predicted time})^2\right]
$$

In [ ]:
def prediction_mse(model, data):
    predictions = model.pointing_time(data["distance"], data["width"])
    return float(np.mean((data["pointing_time"] - predictions) ** 2))


manual_mse = prediction_mse(ExampleModel(a=0.01, b=0.2), experiments_data)
print(f"Manual MSE: {manual_mse:.6f} seconds²")

In the previous task, you tried different values of $a$ and $b$ and inspected the resulting predictions. Now that we have MSE to measure prediction error, we can automate this process. [Hyperopt](https://hyperopt.github.io/hyperopt/#hyperopt-distributed-asynchronous-hyper-parameter-optimization) is a library for finding parameter values that minimize an objective function. 
For our model, we specify:
- An objective function: the MSE between the model’s predictions and the measured pointing times.
- A search space: the range of values to explore for $a$ and $b$.

The code below evaluates 1,000 parameter combinations and returns the best values it finds. 
Check out [the basic tutorial](https://hyperopt.github.io/hyperopt/tutorials/01.BasicTutorial/) for a walkthrough of these steps.

After tuning the parameters, examine the predictions again: do they capture the patterns in the data, or might we need a different model?


In [ ]:
# Define an objective function using MSE as the loss.
def objective(params):
    return prediction_mse(ExampleModel(**params), experiments_data)

# Define a search space using uniform distributions for the continuous parameters a and b.
space =  {
        "a": hp.uniform("a", 0, 0.1),
        "b": hp.uniform("b", 0, 1),
    }

baseline_params = fmin(
    fn=objective,   # Objective function to minimize
    space=space,    # Parameter search space
    algo=tpe.suggest,  # Optimization algorithm: TPE
    max_evals=1000, # Number of parameter combinations to evaluate
)

print(f"Best parameters: a = {baseline_params['a']:.4f}, b = {baseline_params['b']:.4f}")
plot_predictions(ExampleModel(**baseline_params), experiments_data)

### Task 4: Fit and compare candidate models with Hyperopt (~25 minutes)

Even after using Hyperopt to tune $a$ and $b$, you may notice that the model’s predictions still do not closely match the measured times. Now it’s your turn to explore other models. 

Implement the candidate models below and use the same Hyperopt workflow to tune their parameters. 
For each model, plot its predictions alongside the measured movement times. Then explain which model best fits the data.


The two models below predict movement time $T$ from target distance $D$ and width $W$:

**1. Logarithm**

$$
T = a\ln\left(\frac{D}{W}\right) + b
$$

**2. Logarithm with an offset**

$$
T = a\log_2\left(1 + \frac{D}{W}\right) + b
$$

Use `np.log` and `np.log2`. 

In [ ]:
# TODO
